# 2Q iSWAP — asymmetric anharmonicities  (warm-start for 3Q)

Solves the 4-dim 2Q resonant-exchange iSWAP problem with **per-qubit anharmonicities**:

- `η_1 / 2π = −183.6 MHz`  (qubit 1)
- `η_2 / 2π = −181.2 MHz`  (qubit 2)

Different from the symmetric `iswap_pulse_construction.ipynb` (which used η = −170 MHz for both). Each qubit gets its **own** `SpectralLeakageConstraintIQ` at its own `|η_i|` so the optimizer suppresses each qubit's 1Q leakage channel independently.

The resulting 4-dim trajectory is intended as a warm-start for a subsequent 3-qubit optimization. Save format is compatible with the polish-script loader.

In [ ]:
import Pkg
Pkg.activate(@__DIR__)
piccolo_path       = joinpath(@__DIR__, "..", "..", "..", "Piccolo.jl")
directtrajopt_path = joinpath(@__DIR__, "..", "..", "..", "DirectTrajOpt.jl")
Pkg.develop([
    Pkg.PackageSpec(path = piccolo_path),
    Pkg.PackageSpec(path = directtrajopt_path),
])
Pkg.instantiate()

using Piccolo, FFTW, CairoMakie, Printf, LinearAlgebra, Random, SparseArrays, JLD2
using DirectTrajOpt.Constraints: AbstractNonlinearConstraint
using DirectTrajOpt.CommonInterface

## Constants

Per-qubit anharmonicities are the only physical change from the symmetric notebook.

In [ ]:
# --- timing layout ---
const T_TOTAL_NS    = 250.0
const T_PULSE_NS    = 200.0
const T_RISE        = 25.0
const T_FALL        = T_RISE + T_PULSE_NS
const T_EDGE_MARK   = 10.0
const T_FLAT_NS     = T_FALL - T_RISE - 2 * T_EDGE_MARK    # 180 ns
const N_FINE        = 16384
const dt_fine       = T_TOTAL_NS / N_FINE

# --- per-qubit anharmonicities (asymmetric) ---
const η_1_MHz       = 183.6                  # qubit 1
const η_2_MHz       = 181.2                  # qubit 2
const η_1_rad_pos   = 2π * η_1_MHz * 1e-3    # +|η_1| in rad/ns, used for spectral-constraint frequency
const η_2_rad_pos   = 2π * η_2_MHz * 1e-3
const η_1_rad_neg   = -2π * η_1_MHz * 1e-3   # transmon convention (η < 0), used for H_anh
const η_2_rad_neg   = -2π * η_2_MHz * 1e-3

# --- envelope filter ---
const B_AWG_MHz     = 250.0
const σ_f_anh_MHz   = 100.0                  # η-killer Gaussian width
const σ_f_AWG_MHz   = B_AWG_MHz / sqrt(log(2))

# --- drive / coupling ---
const g_max_MHz         = 2.0
const g_max_rad_per_ns  = 2π * g_max_MHz * 1e-3
const g_flat            = g_max_rad_per_ns
const g_max_env         = 1.0

# --- MW + optimization ---
const a_bound_MW    = 2π * 0.01
const F_THRESHOLD   = 0.9999
const Q_R           = 1e2
const ε_MAX         = 1e-2
const N_KNOTS       = 15
const N_SAMPLES_OPT = 300
const Δt_FLAT       = T_FLAT_NS / (N_KNOTS - 1)
const NUM_ITER      = 1000
const SEED          = 42

const RUN_TAG = "asym_2lvl_4dim_geff2MHz_200ns_eta1_$(Int(round(η_1_MHz)))_eta2_$(Int(round(η_2_MHz)))"

@printf("η_1 / 2π = %.1f MHz   (rad/ns: +%.4f, neg = %.4f)\n", η_1_MHz, η_1_rad_pos, η_1_rad_neg)
@printf("η_2 / 2π = %.1f MHz   (rad/ns: +%.4f, neg = %.4f)\n", η_2_MHz, η_2_rad_pos, η_2_rad_neg)
@printf("Filter G(|η_1|) = %.3e   G(|η_2|) = %.3e\n",
    exp(-η_1_MHz^2 / (2 * σ_f_anh_MHz^2)),
    exp(-η_2_MHz^2 / (2 * σ_f_anh_MHz^2)))

## Filtered g_eff envelope

In [ ]:
const ts_env = collect(LinRange(0.0, T_TOTAL_NS, N_FINE))

function gaussian_lowpass(g_t::AbstractVector, dt_ns::Float64, σ_MHz::Float64)
    N      = length(g_t)
    G      = fft(g_t)
    fs_MHz = fftfreq(N, 1.0 / dt_ns) .* 1e3
    mask   = exp.(-0.5 .* (fs_MHz ./ σ_MHz).^2)
    return real.(ifft(G .* mask))
end

g_square      = Float64[(T_RISE ≤ t ≤ T_FALL) ? g_max_env : 0.0 for t in ts_env]
g_after_anh   = gaussian_lowpass(g_square,    dt_fine, σ_f_anh_MHz)
g_after_AWG   = gaussian_lowpass(g_after_anh, dt_fine, σ_f_AWG_MHz)
g_phys_env    = g_after_AWG .* g_max_rad_per_ns

const cumul   = cumsum(g_phys_env) .* dt_fine
const total_g = cumul[end]

flat_start_idx = argmin(abs.(ts_env .- (T_RISE + T_EDGE_MARK)))
flat_end_idx   = argmin(abs.(ts_env .- (T_FALL - T_EDGE_MARK)))

A_rise        = cumul[flat_start_idx]
A_fall        = cumul[end] - cumul[flat_end_idx]
A_flat_drift  = cumul[flat_end_idx] - cumul[flat_start_idx]

@printf("∫ g_eff dt total = %.4f rad = %.3f · (π/4)\n", total_g, total_g / (π/4))
@printf("A_rise = %.4f rad,  A_fall = %.4f rad,  A_flat = %.4f rad\n", A_rise, A_fall, A_flat_drift)

## Pauli operators (2Q computational subspace)

In [ ]:
const XX = operator_from_string("XX")
const YY = operator_from_string("YY")
const XI = operator_from_string("XI")
const YI = operator_from_string("YI")
const IX = operator_from_string("IX")
const IY = operator_from_string("IY")
const ZI = operator_from_string("ZI")
const IZ = operator_from_string("IZ")
const ZZ = operator_from_string("ZZ")

## Edge unitaries and flat-region target

Closed-form `(XX+YY)` rotations from the integrated edge areas. Same construction as the symmetric notebook.

In [ ]:
V_rise   = exp(-im * A_rise * (XX + YY))
V_fall   = exp(-im * A_fall * (XX + YY))
U_iSWAP  = exp(-im * (π/4) * (XX + YY))
U_goal   = V_fall' * U_iSWAP * V_rise'

F_check  = abs2(tr(U_iSWAP' * (V_fall * U_goal * V_rise))) / 16
@printf("F(V_fall · U_goal · V_rise, U_iSWAP) = %.10f\n", F_check)

## SpectralLeakageConstraintIQ — per-qubit IQ-pair version

Identical to `iswap_pulse_construction.ipynb`. Each qubit's IQ pair (`u_X1, u_Y1`) or (`u_X2, u_Y2`) is constrained at its own qubit's `|η|` frequency.

In [ ]:
struct SpectralLeakageConstraintIQ <: AbstractNonlinearConstraint
    name::Symbol
    i_X::Int
    i_Y::Int
    ω::Float64
    ε_max::Float64
    times_t::Vector{Float64}
    Δts::Vector{Float64}
    cosωt::Vector{Float64}
    sinωt::Vector{Float64}
    dim::Int
    equality::Bool
end

function SpectralLeakageConstraintIQ(name::Symbol, i_X::Int, i_Y::Int,
                                     ω::Float64, ε_max::Float64, traj)
    times = get_times(traj)
    Δts   = get_timesteps(traj)
    return SpectralLeakageConstraintIQ(name, i_X, i_Y, ω, ε_max,
        Vector{Float64}(times), Vector{Float64}(Δts),
        cos.(ω .* times), sin.(ω .* times),
        1, false)
end

function _ReIm_IQ(c::SpectralLeakageConstraintIQ, u::AbstractMatrix)
    Re_sum = zero(eltype(u)); Im_sum = zero(eltype(u))
    @inbounds for k in 1:length(c.times_t)
        cos_k = c.cosωt[k]; sin_k = c.sinωt[k]; Δt_k = c.Δts[k]
        uX = u[c.i_X, k]; uY = u[c.i_Y, k]
        Re_sum += Δt_k * (uX * cos_k + uY * sin_k)
        Im_sum += Δt_k * (uY * cos_k - uX * sin_k)
    end
    return Re_sum, Im_sum
end

function CommonInterface.evaluate!(values::AbstractVector,
                                   c::SpectralLeakageConstraintIQ, traj)
    u = traj[c.name]
    Re_sum, Im_sum = _ReIm_IQ(c, u)
    values[1] = Re_sum^2 + Im_sum^2 - c.ε_max
    return nothing
end

function CommonInterface.eval_jacobian(c::SpectralLeakageConstraintIQ, traj)
    Z_dim = traj.dim * traj.N + traj.global_dim
    ∂g    = spzeros(c.dim, Z_dim)
    u     = traj[c.name]
    Re_sum, Im_sum = _ReIm_IQ(c, u)
    comps = traj.components[c.name]
    @inbounds for k in 1:traj.N
        cos_k = c.cosωt[k]; sin_k = c.sinωt[k]; Δt_k = c.Δts[k]
        idxX = traj.dim * (k - 1) + comps[c.i_X]
        idxY = traj.dim * (k - 1) + comps[c.i_Y]
        ∂g[1, idxX] = 2 * Δt_k * (Re_sum * cos_k - Im_sum * sin_k)
        ∂g[1, idxY] = 2 * Δt_k * (Re_sum * sin_k + Im_sum * cos_k)
    end
    return ∂g
end

function CommonInterface.eval_hessian_of_lagrangian(c::SpectralLeakageConstraintIQ,
                                                    traj, μ::AbstractVector)
    Z_dim = traj.dim * traj.N + traj.global_dim
    ∂²g   = spzeros(Z_dim, Z_dim)
    comps = traj.components[c.name]
    a = zeros(Z_dim); b = zeros(Z_dim)
    @inbounds for k in 1:traj.N
        idxX = traj.dim * (k - 1) + comps[c.i_X]
        idxY = traj.dim * (k - 1) + comps[c.i_Y]
        a[idxX] = c.Δts[k] * c.cosωt[k]
        a[idxY] = c.Δts[k] * c.sinωt[k]
        b[idxX] = -c.Δts[k] * c.sinωt[k]
        b[idxY] =  c.Δts[k] * c.cosωt[k]
    end
    nzs = findall(!iszero, a) ∪ findall(!iszero, b)
    for i in nzs, j in nzs
        v = 2 * μ[1] * (a[i]*a[j] + b[i]*b[j])
        if v != 0.0
            ∂²g[i, j] = v
        end
    end
    return ∂²g
end

println("SpectralLeakageConstraintIQ defined.")

## Flat-region optimization problem

The only difference from the symmetric notebook is that the two `SpectralLeakageConstraintIQ` instances now use **different** ω values (qubit 1's and qubit 2's anharmonicities).

In [ ]:
function H_flat_gate(u, t)
    H = g_flat * (XX + YY)
    H += u[1] * XI + u[2] * YI
    H += u[3] * IX + u[4] * IY
    return H
end

H_vars = Function[
    (u, t) -> ZI,
    (u, t) -> IZ,
    (u, t) -> ZZ,
]

drive_bounds = fill(a_bound_MW, 4)

varsys = VariationalQuantumSystem(
    H_flat_gate, H_vars, 4, drive_bounds; time_dependent = true,
)
@printf("varsys: levels=%d  n_drives=%d  n_vars=%d\n",
    varsys.levels, varsys.n_drives, length(varsys.G_vars))

In [ ]:
Random.seed!(SEED)
controls_init = 2 .* a_bound_MW .* rand(4, N_SAMPLES_OPT) .- a_bound_MW
times_init    = collect(LinRange(0.0, T_FLAT_NS, N_SAMPLES_OPT))
du_init       = zeros(4, N_SAMPLES_OPT)
pulse_init    = CubicSplinePulse(controls_init, du_init, times_init)

qcp = VariationalSplinePulseProblem(
    varsys, pulse_init, U_goal, N_KNOTS;
    Q                     = 0.0,
    Q_r                   = Q_R,
    R                     = 1e-3,
    du_bound              = Inf,
    Δt_bounds             = (Δt_FLAT, Δt_FLAT),
    dynamics_spline_order = 3,
    n_path_samples        = 3,
)

push!(qcp.prob.constraints,
    FinalUnitaryFidelityConstraint(U_goal, :Ũ⃗, F_THRESHOLD, get_trajectory(qcp)))

traj = get_trajectory(qcp)

# Per-qubit spectral leakage constraints — TWO different ω
push!(qcp.prob.constraints,
    SpectralLeakageConstraintIQ(:u, 1, 2, η_1_rad_pos, ε_MAX, traj))   # qubit 1 at |η_1|
push!(qcp.prob.constraints,
    SpectralLeakageConstraintIQ(:u, 3, 4, η_2_rad_pos, ε_MAX, traj))   # qubit 2 at |η_2|

@printf("\nProblem built.\n")
@printf("  N_KNOTS = %d   Δt = %.4f ns\n", N_KNOTS, Δt_FLAT)
@printf("  ε_max   = %.0e per qubit\n", ε_MAX)
@printf("  ω_1     = 2π · %.1f MHz   (qubit 1, η_1)\n", η_1_MHz)
@printf("  ω_2     = 2π · %.1f MHz   (qubit 2, η_2)\n", η_2_MHz)
@printf("  F_thr   = %.4f\n", F_THRESHOLD)
@printf("  Q_r     = %.0e\n", Q_R)

## Solve and save

In [ ]:
const IPOPT_LOG = joinpath(@__DIR__, "ipopt_$(RUN_TAG).log")
const TRAJ_PATH = joinpath(@__DIR__, "traj_$(RUN_TAG).jld2")

println("Solving asymmetric-η 2Q iSWAP...")
println("  IPOPT log:  $IPOPT_LOG")
println("  Trajectory: $TRAJ_PATH")

t0 = time()
try
    solve!(qcp; max_iter = NUM_ITER, print_level = 5,
        options = IpoptOptions(
            eval_hessian    = false,
            output_file     = IPOPT_LOG,
            constr_viol_tol = 1e-8,
            tol             = 1e-8,
            acceptable_tol  = 1e-8,
        ))
catch e
    e isa InterruptException || rethrow(e)
    println("Interrupted")
end
@printf("solve wall: %.1f s\n", time() - t0)

traj_robust  = get_trajectory(qcp)
U_flat_final = iso_vec_to_operator(traj_robust[:Ũ⃗][:, end])
F_flat       = abs2(tr(U_goal' * U_flat_final)) / 16
U_full       = V_fall * U_flat_final * V_rise
F_full       = abs2(tr(U_iSWAP' * U_full)) / 16

@printf("\nF(U_flat, U_goal)              = %.8f   (1−F = %.3e)\n", F_flat, 1 - F_flat)
@printf("F(V_fall·U_flat·V_rise, U_iSWAP) = %.8f   (1−F = %.3e)\n", F_full, 1 - F_full)

@save TRAJ_PATH traj_robust U_goal U_iSWAP V_rise V_fall A_rise A_fall A_flat_drift g_max_MHz η_1_MHz η_2_MHz T_FLAT_NS T_RISE T_FALL T_EDGE_MARK
@printf("\nSaved: %s\n", TRAJ_PATH)

## Next

This trajectory is the warm-start for the 3-qubit optimization. Save format mirrors the symmetric `iswap_pulse_construction.ipynb` so the polish-script loader works with minimal changes (just point `TRAJ_PATH` and read per-qubit η).

For honest 3-lvl verification of this 2Q gate before scaling to 3Q, use the polish-script propagator with per-qubit anharmonicities in `H_anh_9`:

```julia
H_anh_9 = (η_1_rad_neg / 2) * (n1_9 * (n1_9 - I_9)) +
          (η_2_rad_neg / 2) * (n2_9 * (n2_9 - I_9))
```

That replaces the symmetric `H_anh_9 = (η_rad / 2) * (n1_9*(n1_9-I_9) + n2_9*(n2_9-I_9))`. Everything else stays.